# EB-JEPA CIFAR-10 comparison workflow

Run the shared imports once, then edit and run the parameter and execution cells for the section you need. Evaluate and Benchmark intentionally update the same per-run `result.json`; Report consumes the six completed run artifacts.

In [ ]:
#!git clone https://github.com/Baptistecaille/ReccurentCorticalColumn.git
from pathlib import Path

PROJECT_ROOT = Path("/Users/baptistecaillerie/Documents/ReccurentCorticalColumn/eb_jepa_cifar10_comparison") #Path("/content/ReccurentCorticalColumn/eb_jepa_cifar10_comparison")
SOURCE_ROOT = PROJECT_ROOT / "src"
DATA_ROOT = PROJECT_ROOT / "data"
RUNS_ROOT = PROJECT_ROOT / "runs"
REPORTS_ROOT = PROJECT_ROOT / "reports"

#%cd {SOURCE_ROOT}
#!uv sync
#!pip install fvcore

In [2]:
!mkdir -p {DATA_ROOT}
!wget -q --show-progress \
  -O {DATA_ROOT}/cifar-10-python.tar.gz \
  "https://huggingface.co/Peyiloo/peyiloo/resolve/main/cifar-10-python.tar.gz?download=true"

!echo "c58f30108f718f92721af3b95e74349a  {DATA_ROOT}/cifar-10-python.tar.gz" | md5sum -c -
!tar -xzf {DATA_ROOT}/cifar-10-python.tar.gz -C {DATA_ROOT}


/content/ReccurentC 100%[===================>] 162.60M   334MB/s    in 0.5s    
/content/ReccurentCorticalColumn/eb_jepa_cifar10_comparison/data/cifar-10-python.tar.gz: OK


In [8]:
from IPython.display import display
from comparison.config import load_config
from comparison.train import run
from comparison.workflow import benchmark_checkpoint_to_json
from comparison.workflow import evaluate_checkpoint_to_json
from comparison.workflow import report_from_json

In [9]:
print(f"Working directory: {Path.cwd()}")
print(f"Run directory: {RUNS_ROOT}")
for generated_path in sorted(RUNS_ROOT.rglob("*")):
    if generated_path.is_file():
        print(generated_path.relative_to(PROJECT_ROOT))

Working directory: /Users/baptistecaillerie/Documents/ReccurentCorticalColumn/eb_jepa_cifar10_comparison/scripts
Run directory: /content/ReccurentCorticalColumn/eb_jepa_cifar10_comparison/runs


## Train

Choose one architecture configuration and seed. The execution cell displays the best checkpoint produced in the run directory.

In [10]:
TRAIN_CONFIG = PROJECT_ROOT / "configs" / "cortical.yaml"
TRAIN_OVERRIDES = [f"data.root={DATA_ROOT}"]
TRAIN_SEED = 1
TRAIN_OUTPUT_DIR = RUNS_ROOT / "cortical" / str(TRAIN_SEED)
TRAIN_RESUME_FROM = None

In [11]:
train_cfg = load_config(TRAIN_CONFIG, TRAIN_OVERRIDES)
train_result = run(train_cfg, TRAIN_SEED, TRAIN_OUTPUT_DIR, TRAIN_RESUME_FROM)
display(train_result.best_checkpoint)

FileNotFoundError: Config file not found: /content/ReccurentCorticalColumn/eb_jepa_cifar10_comparison/configs/cortical.yaml

## Evaluate

Evaluate an existing best checkpoint with exactly five pair seeds. The returned path is the shared artifact for this architecture and training seed.

In [ ]:
EVALUATE_CONFIG = PROJECT_ROOT / "configs" / "cortical.yaml"
EVALUATE_OVERRIDES = [f"data.root={DATA_ROOT}"]
EVALUATE_CHECKPOINT = RUNS_ROOT / "cortical" / "1" / "best.pt"
EVALUATE_PAIR_SEEDS = (11, 22, 33, 44, 55)
EVALUATE_OUTPUT = RUNS_ROOT / "cortical" / "1" / "result.json"

In [23]:
evaluation_cfg = load_config(EVALUATE_CONFIG, EVALUATE_OVERRIDES)
evaluation_path = evaluate_checkpoint_to_json(
    evaluation_cfg, EVALUATE_CHECKPOINT, EVALUATE_PAIR_SEEDS, EVALUATE_OUTPUT
)
display(evaluation_path)

PosixPath('/content/ReccurentCorticalColumn/eb_jepa_cifar10_comparison/runs/baseline/1/result.json')

## Benchmark

Benchmark the same checkpoint on the protocol-required A100. This updates and displays the same per-run artifact path used by Evaluate.

In [ ]:
BENCHMARK_CONFIG = PROJECT_ROOT / "configs" / "cortical.yaml"
BENCHMARK_OVERRIDES = [f"data.root={DATA_ROOT}"]
BENCHMARK_CHECKPOINT = RUNS_ROOT / "cortical" / "1" / "best.pt"
BENCHMARK_OUTPUT = RUNS_ROOT / "cortical" / "1" / "benchmark.json"

In [22]:
benchmark_cfg = load_config(BENCHMARK_CONFIG, BENCHMARK_OVERRIDES)
benchmark_path = benchmark_checkpoint_to_json(
    benchmark_cfg, BENCHMARK_CHECKPOINT, BENCHMARK_OUTPUT
)
display(benchmark_path)

PosixPath('/content/ReccurentCorticalColumn/eb_jepa_cifar10_comparison/runs/baseline/1/benchmark.json')

## Report

Provide three completed Baseline artifacts and three completed Cortical artifacts. The execution cell displays both report paths and the final comparison decision.

In [26]:
REPORT_RESULTS = (
    RUNS_ROOT / "baseline" / "1" / "result.json",
    RUNS_ROOT / "baseline" / "1000" / "result.json",
    RUNS_ROOT / "baseline" / "10000" / "result.json",
    RUNS_ROOT / "cortical" / "1" / "result.json",
    RUNS_ROOT / "cortical" / "1000" / "result.json",
    RUNS_ROOT / "cortical" / "10000" / "result.json",
)
REPORT_MARKDOWN_OUTPUT = REPORTS_ROOT / "comparison.md"
REPORT_JSON_OUTPUT = REPORTS_ROOT / "comparison.json"
REPORT_SCORE_TOLERANCE = 0.02

In [27]:
report_result = report_from_json(
    REPORT_RESULTS,
    REPORT_MARKDOWN_OUTPUT,
    REPORT_JSON_OUTPUT,
    REPORT_SCORE_TOLERANCE,
)
display(report_result.markdown_path)
display(report_result.json_path)
display(report_result.decision)

FileNotFoundError: [Errno 2] No such file or directory: '/content/ReccurentCorticalColumn/eb_jepa_cifar10_comparison/runs/baseline/1000/result.json'